In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

df = pd.read_csv("../../data/_data.csv")

In [2]:
# Оставляем только те колонки, которые относятся ко мне

df = df[["ID  объявления","Метро", "Площадь, м2", "Цена", "Высота потолков, м"]]

df = df.rename(columns={"ID  объявления": "id"})

df.head()

,id,Метро,"Площадь, м2",Цена,"Высота потолков, м"
0,271271157,м. Смоленская (9 мин пешком),200.0/20.0,"500000.0 руб./ За месяц, Залог - 500000 руб., ...",3.0
1,271634126,м. Смоленская (8 мин пешком),198.0/95.0/18.0,"500000.0 руб./ За месяц, Залог - 500000 руб., ...",3.5
2,271173086,м. Смоленская (7 мин пешком),200.0/116.0/4.0,"500000.0 руб./ За месяц, Залог - 500000 руб., ...",3.2
3,272197456,м. Смоленская (3 мин пешком),170.0/95.0/17.0,"400000.0 руб./ За месяц, Залог - 400000 руб., ...",3.2
4,273614615,м. Арбатская (7 мин пешком),58.0/38.0/5.0,"225000.0 руб./ За месяц, Залог - 225000 руб., ...",3.9


In [3]:
# Метро

# Создаем колонки на основе исходных данных
df['metro_minutes'] = df['Метро'].str.extract(r'(\d+)').fillna(0).astype(int)
df['metro_walking'] = (df['Метро'].str.extract(r'(пешком|на машине)').fillna('пешком')[0] == 'пешком').astype(int)

# Удаляем исходную колонку Метро, так как мы извлекли из нее все нужное
df = df.drop('Метро', axis=1)

# Считаем количество нулей (значения, где не было цифр)
zeros_count = (df['metro_minutes'] == 0).sum()
print(f"Всего нулей (где время не указано): {zeros_count}")

# Вычисляем среднее время отдельно для пешком и для машины
# Округляем до целых чисел, чтобы сохранить тип int
avg_walking = int(round(df[df['metro_walking'] == 1]['metro_minutes'].mean()))
avg_car = int(round(df[df['metro_walking'] == 0]['metro_minutes'].mean()))

print(f"Среднее время пешком (округлено): {avg_walking} мин")
print(f"Среднее время на машине (округлено): {avg_car} мин")

# Заменяем нули на средние значения (с учетом типа передвижения)
# TODO ЗАПОЛНЯЕМ ПРОПУСКИ
# df.loc[(df['metro_minutes'] == 0) & (df['metro_walking'] == 1), 'metro_minutes'] = avg_walking
# df.loc[(df['metro_minutes'] == 0) & (df['metro_walking'] == 0), 'metro_minutes'] = avg_car

# Проверяем, что все нули заменены
print(f"\nОсталось нулей после замены: {(df['metro_minutes'] == 0).sum()}")

# Смотрим итоговую статистику
print("\nНовая статистика:")
print(df[['metro_minutes', 'metro_walking']].describe())

Всего нулей (где время не указано): 2453
Среднее время пешком (округлено): 11 мин
Среднее время на машине (округлено): 10 мин

Осталось нулей после замены: 2453

Новая статистика:
       metro_minutes  metro_walking
count   23368.000000   23368.000000
mean       11.039670       0.882831
std        63.582358       0.321628
min         0.000000       0.000000
25%         4.000000       1.000000
50%         8.000000       1.000000
75%        13.000000       1.000000
max      1905.000000       1.000000


In [4]:
# Площадь

area_cols = ['total_area', 'living_area', 'kitchen_area']
df[area_cols] = df['Площадь, м2'].str.split('/', expand=True).apply(pd.to_numeric, errors='coerce')

# Заполняем пропуски
# Для жилой площади и кухни - 0 (это логично для студий)
# TODO ЗАПОЛНЯЕМ ПРОПУСКИ
# df['living_area'] = df['living_area'].fillna(0)
# df['kitchen_area'] = df['kitchen_area'].fillna(0)

# Удаляем исходную колонку Площадь, так как мы извлекли из нее все нужное
df = df.drop('Площадь, м2', axis=1)

In [5]:
# Высота потолков

df = df.rename(columns={"Высота потолков, м": "ceiling_height"})

# Исходные данные
heights= df['ceiling_height']

# Автоматически рассчитываем перцентили
valid_heights = heights.dropna()
LOWER = valid_heights.quantile(0.01)  # 1-й перцентиль
UPPER = valid_heights.quantile(0.99)  # 99-й перцентиль
MEDIAN = valid_heights.median()       # медиана

# Находим выбросы
outliers_mask = (heights < LOWER) | (heights > UPPER)

# Создаем очищенную версию (выбросы -> NaN)
cleaned = heights.copy()
cleaned[outliers_mask] = np.nan

# Заполняем все NaN (и выбросы, и исходные пропуски)
# TODO ЗАПОЛНЯЕМ ПРОПУСКИ
# df['ceiling_height'] = cleaned.fillna(MEDIAN)

# Печатаем статистику для проверки
print(f"Границы по перцентилям:")
print(f"  1% = {LOWER:.2f} м")
print(f"  99% = {UPPER:.2f} м")
print(f"  Медиана = {MEDIAN:.2f} м")
print(f"\nНайдено выбросов: {outliers_mask.sum()}")
print(f"Исходных пропусков: {heights.isna().sum()}")
print(f"\nРезультат:")

Границы по перцентилям:
  1% = 2.48 м
  99% = 4.00 м
  Медиана = 2.64 м

Найдено выбросов: 108
Исходных пропусков: 12162

Результат:


In [6]:
# Цена

# Исходные данные
df['price'] = df['Цена'].str.extract(r'([\d.]+)\s*руб\./\s*За месяц').astype(float)

# 1. Коммуналка (сумма)
df['utilities_amount'] = df['Цена'].str.extract(r'Сумма коммунальных платежей\s*-\s*([\d.]+)').astype(float)

# 2. Коммуналка включена (флаг)
df['utilities_included'] = df['Цена'].str.contains('Коммунальные услуги включены', na=False).astype(int)

# 3. Тип срока аренды
df['rental_term'] = df['Цена'].str.extract(r'Срок аренды\s*-\s*([^,]+)')[0].str.strip()

# 4. Предоплата (месяцы)
df['prepayment_months'] = df['Цена'].str.extract(r'Предоплата\s*(\d+)\s*мес').astype(float)

# Удаляем исходную колонку Цена, так как мы извлекли из нее все нужное
df = df.drop('Цена', axis=1)

# Статистика
print("\nСтатистика:")
print(f"Цена: {df['price'].notna().sum()}/{len(df)}")
print(f"Сумма коммуналки: {df['utilities_amount'].notna().sum()}/{len(df)}")
print(f"Коммуналка включена: {df['utilities_included'].sum()}/{len(df)}")
print(f"Срок аренды: {df['rental_term'].notna().sum()}/{len(df)}")
print(f"Предоплата: {df['prepayment_months'].notna().sum()}/{len(df)}")


Статистика:
Цена: 23344/23368
Сумма коммуналки: 2562/23368
Коммуналка включена: 17745/23368
Срок аренды: 23368/23368
Предоплата: 22777/23368


In [7]:
# 1) Сначала исправляем противоречия: если есть сумма >0 → коммуналка НЕ включена
mask_has_amount = df['utilities_amount'].notna() & (df['utilities_amount'] > 0)
df.loc[mask_has_amount, 'utilities_included'] = 0

# 2) Если нет суммы коммуналки (NaN или 0) → utilities_included=1 и amount=0
mask_no_amount = df['utilities_amount'].isna() | (df['utilities_amount'] == 0)
df.loc[mask_no_amount, ['utilities_included', 'utilities_amount']] = [1, 0]

# 3) rental_term → is_long_rental_term
df['is_long_rental_term'] = (df['rental_term'] == 'Длительный').astype(int)

# TODO ЗАПОЛНЯЕМ ПРОПУСКИ
# 4) prepayment_months заполняем модой
# df['prepayment_months'] = df['prepayment_months'].fillna(df['prepayment_months'].mode()[0])

# 5) Удаляем старую колонку rental_term
df = df.drop('rental_term', axis=1)

# Проверка
df[['utilities_amount', 'utilities_included', 'is_long_rental_term', 'prepayment_months']].head()

,utilities_amount,utilities_included,is_long_rental_term,prepayment_months
0,0.0,1,1,1.0
1,0.0,1,1,1.0
2,0.0,1,1,1.0
3,0.0,1,1,1.0
4,0.0,1,1,1.0


In [8]:
df.to_csv('../../tmp_data/bek_processed_part_1.csv', index=False)
df.head()

,id,ceiling_height,metro_minutes,metro_walking,total_area,living_area,kitchen_area,price,utilities_amount,utilities_included,prepayment_months,is_long_rental_term
0,271271157,3.0,9,1,200.0,20.0,NaN,500000.0,0.0,1,1.0,1
1,271634126,3.5,8,1,198.0,95.0,18.0,500000.0,0.0,1,1.0,1
2,271173086,3.2,7,1,200.0,116.0,4.0,500000.0,0.0,1,1.0,1
3,272197456,3.2,3,1,170.0,95.0,17.0,400000.0,0.0,1,1.0,1
4,273614615,3.9,7,1,58.0,38.0,5.0,225000.0,0.0,1,1.0,1
